## 1. OVERVIEW & PREPROCESSING

In [ ]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHoOrdinalEncodertEncoder, 

### 1.1. Dataset Overview

In [3]:
train_df = pd.read_csv("dataset/raw/train.csv")
test_df  = pd.read_csv("dataset/raw/test.csv")

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2492196 entries, 0 to 2492195
Data columns (total 28 columns):
 #   Column             Dtype  
---  ------             -----  
 0   ID                 object 
 1   Severity           int64  
 2   Start_Time         object 
 3   End_Time           object 
 4   Latitude           float64
 5   Longitude          float64
 6   Distance(mi)       float64
 7   Description        object 
 8   Street             object 
 9   City               object 
 10  County             object 
 11  State              object 
 12  Zipcode            object 
 13  Timezone           object 
 14  Airport_Code       object 
 15  Weather_Timestamp  object 
 16  Temperature(F)     float64
 17  Humidity(%)        float64
 18  Visibility(mi)     float64
 19  Weather_Condition  object 
 20  Amenity            bool   
 21  Crossing           bool   
 22  Junction           bool   
 23  Railway            bool   
 24  Station            bool   
 25  Stop              

In [98]:
train_df.head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,212018647,12/24/2021 12:00:00 AM,12/24/2021 12:00:00 AM,1115,20,Olympic,2019,1,510,VEHICLE - STOLEN,...,IC,Invest Cont,510.0,NaN,NaN,NaN,100 S VERMONT AV,NaN,34.0721,-118.2917
1,201107812,03/28/2020 12:00:00 AM,03/28/2020 12:00:00 AM,2055,11,Northeast,1149,1,210,ROBBERY,...,IC,Invest Cont,210.0,NaN,NaN,NaN,5100 N FIGUEROA ST,NaN,34.1056,-118.2008
2,240708269,04/20/2024 12:00:00 AM,04/19/2024 12:00:00 AM,1900,7,Wilshire,734,2,745,VANDALISM - MISDEAMEANOR ($399 OR UNDER),...,IC,Invest Cont,745.0,NaN,NaN,NaN,400 S CURSON AV,NaN,34.0675,-118.3536
3,210612806,07/16/2021 12:00:00 AM,07/15/2021 12:00:00 AM,2030,6,Hollywood,659,1,330,BURGLARY FROM VEHICLE,...,IC,Invest Cont,330.0,NaN,NaN,NaN,1300 N KINGSLEY DR,NaN,34.0951,-118.3031
4,211410554,04/27/2021 12:00:00 AM,04/27/2021 12:00:00 AM,1145,14,Pacific,1414,2,901,VIOLATION OF RESTRAINING ORDER,...,AO,Adult Other,901.0,NaN,NaN,NaN,600 SAN JUAN AV,NaN,33.9932,-118.4671


In [18]:
train_df.isnull().sum()

ID                       0
Severity                 0
Start_Time               0
End_Time                 0
Latitude                 0
Longitude                0
Distance(mi)             0
Description              0
Street                5411
City                   100
County                   0
State                    0
Zipcode                724
Timezone              2782
Airport_Code          9295
Weather_Timestamp    47950
Temperature(F)       62100
Humidity(%)          65945
Visibility(mi)       64006
Weather_Condition    62172
Amenity                  0
Crossing                 0
Junction                 0
Railway                  0
Station                  0
Stop                     0
Traffic_Signal           0
Sunrise_Sunset       12720
dtype: int64

In [5]:
def show_unique_values(df, columns=None, max_display=50):
    """
    Display the unique values of columns in a DataFrame
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The DataFrame to analyze
    columns : list, optional
        A list of columns to display. If None, all columns will be displayed
    max_display : int, default=50
        The maximum number of unique values to display for each column
        
    Returns:
    --------
    dict
        A dictionary containing unique values for each column
    """
    # If columns are not specified, take all columns
    if columns is None:
        columns = df.columns.tolist()
    
    # Dictionary to store results
    unique_dict = {}
    
    print("=" * 100)
    print("UNIQUE VALUES ANALYSIS")
    print("=" * 100)
    
    for col in columns:
        if col not in df.columns:
            print(f"Column '{col}' does not exist in the DataFrame")
            continue
        
        unique_vals = df[col].unique()
        n_unique = len(unique_vals)
        
        # Get value counts
        value_counts = df[col].value_counts(dropna=False)
        
        # Store in the dictionary
        unique_dict[col] = unique_vals
        
        # Print information
        print(f"\nFeature: {col}")
        print(f"   • Data type: {df[col].dtype}")
        print(f"   • Total unique values: {n_unique}")
        print(f"   • Missing values: {df[col].isnull().sum()} ({df[col].isnull().sum()/len(df)*100:.2f}%)")
        
        # Display unique values
        if n_unique <= max_display:
            print(f"   • Unique values: {sorted([str(v) for v in unique_vals if pd.notna(v)])}")
            print(f"\n   Value Counts:")
            for val, count in value_counts.head(20).items():
                pct = (count / len(df)) * 100
                print(f"      {str(val):<30} : {count:>6} ({pct:>5.2f}%)")
        else:
            print(f"   • Too many unique values ({n_unique}). Showing top 20:")
            for val, count in value_counts.head(20).items():
                pct = (count / len(df)) * 100
                print(f"      {str(val):<30} : {count:>6} ({pct:>5.2f}%)")
        
        print("-" * 100)
    
    return unique_dict


In [6]:
# Ví dụ 3: Xem unique values của tất cả các cột (cẩn thận với dataset lớn)
unique_all = show_unique_values(train_df)

UNIQUE VALUES ANALYSIS

Feature: ID
   • Data type: object
   • Total unique values: 2492196
   • Missing values: 0 (0.00%)
   • Too many unique values (2492196). Showing top 20:
      A-5010788                      :      1 ( 0.00%)
      A-4081224                      :      1 ( 0.00%)
      A-4338502                      :      1 ( 0.00%)
      A-4813526                      :      1 ( 0.00%)
      A-6175925                      :      1 ( 0.00%)
      A-5238058                      :      1 ( 0.00%)
      A-5792559                      :      1 ( 0.00%)
      A-6685610                      :      1 ( 0.00%)
      A-5466899                      :      1 ( 0.00%)
      A-7452796                      :      1 ( 0.00%)
      A-4851060                      :      1 ( 0.00%)
      A-3708806                      :      1 ( 0.00%)
      A-5822356                      :      1 ( 0.00%)
      A-4896486                      :      1 ( 0.00%)
      A-4942524                      :      1 ( 0.0

### 1.2. Convert to the correct type

In [ ]:
# Convert time columns to datetime

time_col = ["Start_Time", "End_Time", "Weather_Timestamp"]

for col in time_col:
    train_df[col] = pd.to_datetime(train_df[col], errors='coerce')
    test_df[col] = pd.to_datetime(test_df[col], errors='coerce')

In [43]:
# Convert bool columns to object

bool_col = ['Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal']

for col in bool_col:
    train_df[col] = train_df[col].astype('object')
    test_df[col] = test_df[col].astype('object')

### 1.3. Define X and y

In [46]:
X_train = train_df.drop("Severity", axis=1)
y_train = train_df["Severity"]

X_test = test_df.drop("Severity", axis=1)
y_test = test_df["Severity"]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (2492196, 27)
y_train shape: (2492196,)
X_test shape: (623050, 27)
y_test shape: (623050,)


### 1.4. Feature Dropping Strategy

We systematically remove features that cause data leakage, have excessive missing values, or provide redundant/irrelevant information for predicting accident severity.

#### a. Data Leakage (Target Information)
Features that directly reveal the severity outcome must be removed:
- `ID`: Unique identifier with no predictive value
- `Description`: Text containing severity keywords (e.g., "serious accident", "minor incident")
- `End_Time`: Duration (End_Time - Start_Time) directly correlates with severity - more severe accidents take longer to clear
- `Distance(mi)`: Affected road length is a direct consequence of severity, not a predictor

#### b. Redundant Location Features
High-cardinality location features that are too specific or redundant:
- `Street`: Extremely high cardinality (10,000+ unique values); already captured by City/County/State
- `Zipcode`: Overly granular; aggregate location (City/County) is sufficient
- `Latitude` / `Longitude`: Exact coordinates add noise without explanatory power for severity; aggregate location features are more generalizable

#### c. Metadata & Auxiliary Information
Features that provide no predictive signal:
- `Airport_Code`: Weather station identifier; actual weather measurements are already included
- `Timezone`: Redundant with State
- `Weather_Timestamp`: Exact timestamp of weather reading is unnecessary; weather values themselves are what matter

**Total features to drop: 11**

In [77]:
print("=" * 90)
print("FEATURE DROPPING STRATEGY")
print("=" * 90)

drop_leakage = ['ID', 'Description', 'End_Time', 'Distance(mi)']
drop_location = ['Street', 'Zipcode', 'Latitude', 'Longitude']
drop_metadata = ['Airport_Code', 'Timezone', 'Weather_Timestamp']

all_cols_to_drop = drop_leakage + drop_location + drop_metadata

# Filter only existing columns
cols_to_drop = [col for col in all_cols_to_drop if col in X_train.columns]

print(f"\nDropping {len(cols_to_drop)} features from {X_train.shape[1]} total features:\n")

# Apply drops to both train and test sets
X_train = X_train.drop(columns=cols_to_drop, errors='ignore')
X_test = X_test.drop(columns=cols_to_drop, errors='ignore')

print(f"   • Features dropped: {len(cols_to_drop)}")
print(f"   • Train shape: {X_train.shape}")
print(f"   • Test shape:  {X_test.shape}")

FEATURE DROPPING STRATEGY

Dropping 0 features from 16 total features:

   • Features dropped: 0
   • Train shape: (2492196, 16)
   • Test shape:  (623050, 16)


### 1.5. Define Categorical and Numerical Variables


In [64]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

In [65]:
print(f" Number of numerical features: {len(numeric_features)}")
print(f" List: {numeric_features}\n")

print(f" Number of categorical features: {len(categorical_features)}")
print(f" List: {categorical_features}\n")


 Number of numerical features: 3
 List: ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)']

 Number of categorical features: 12
 List: ['City', 'County', 'State', 'Weather_Condition', 'Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset']



### 1.6. Remove Zero Variance Features and Duplicated Rows

In [67]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline

class VarianceThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0):
        self.threshold = threshold
        self.selector = VarianceThreshold(threshold=self.threshold)
    
    def fit(self, X, y=None):
        numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns
        self.numeric_cols = numeric_cols
        self.selector.fit(X[numeric_cols])
        # Save retained numeric columns
        self.retained_cols = numeric_cols[self.selector.get_support()]
        # Save dropped numeric columns
        self.dropped_cols = [col for col in numeric_cols if col not in self.retained_cols]
        print("Columns removed due to zero variance:", self.dropped_cols)
        return self
    
    def transform(self, X):
        cols_to_keep = list(self.retained_cols) + list(X.columns.difference(self.numeric_cols))
        return X[cols_to_keep]


In [68]:
class ConstantAndDuplicateRemover(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Identify constant columns
        self.constant_cols = [col for col in X.columns if X[col].nunique() == 1]
        print("Columns removed due to having only 1 unique value:", self.constant_cols)
        return self
    
    def transform(self, X):
        # Drop constant columns
        X_cleaned = X.drop(columns=self.constant_cols, errors="ignore")
        # Drop duplicate rows
        X_cleaned = X_cleaned.drop_duplicates()
        return X_cleaned

In [69]:
cleaning_pipeline = Pipeline(steps=[
    ("var_thresh", VarianceThresholdSelector(threshold=0)),
    ("const_dup", ConstantAndDuplicateRemover())

])
X_train_cleaned = cleaning_pipeline.fit_transform(X_train.copy())
print(f"Data shape after cleaning: {X_train_cleaned.shape}")
X_test_cleaned = cleaning_pipeline.transform(X_test.copy())
print(f"Data shape after cleaning: {X_test_cleaned.shape}")

Columns removed due to zero variance: []
Columns removed due to having only 1 unique value: []
Data shape after cleaning: (2072286, 16)
Data shape after cleaning: (587535, 16)


c:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


# 2. MISSING VALUES PROCESSING

### 2.1. Detaild Missing Values Analysis

In [70]:
# Detailed analysis by feature groups
print("=" * 90)
print("DETAILED MISSING VALUES ANALYSIS")
print("=" * 90)

# Determine feature type
def get_feature_type(col):
    if col in numeric_features:
        return 'Numerical'
    elif col in categorical_features:
        return 'Categorical'
    else:
        return 'Other'

# Calculate statistics
missing_info = pd.DataFrame({
    'Feature': X_train_cleaned.columns,
    'Feature Type': [get_feature_type(col) for col in X_train_cleaned.columns],
    'Missing Count': X_train_cleaned.isnull().sum().values,
    'Missing %': (X_train_cleaned.isnull().sum() / len(X_train_cleaned) * 100).values,
    'Present Count': X_train_cleaned.notnull().sum().values,
    'Data Type': X_train_cleaned.dtypes.values
})

# Classify by missing level
missing_info['Missing Level'] = pd.cut(
    missing_info['Missing %'],
    bins=[-0.1, 0, 5, 15, 100],
    labels=['None', 'Low (0-5%)', 'Medium (5-15%)', 'High (>15%)']
)

# Display only features with missing values
missing_only = missing_info[missing_info['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("\nFEATURES WITH MISSING VALUES:\n")
print(f"{'Feature':<30} | {'Type':<15} | {'Missing':<12} | {'%':<8} | {'Level':<15}")
print("-" * 95)
for idx, row in missing_only.iterrows():
    print(f"{row['Feature']:<30} | {row['Feature Type']:<15} | {row['Missing Count']:>10,} | {row['Missing %']:>6.2f}% | {row['Missing Level']:<15}")

# Overall statistics
print("\n" + "=" * 90)
print("OVERALL STATISTICS:")
print(f"   • Total observations: {len(X_train_cleaned):,}")
print(f"   • Total features: {len(X_train_cleaned.columns)}")
print(f"   • Features with missing: {len(missing_only)}")
print(f"   • Features without missing: {len(X_train_cleaned.columns) - len(missing_only)}")
print(f"   • Total missing cells: {X_train_cleaned.isnull().sum().sum():,}")
print(f"   • Overall missing rate: {(X_train_cleaned.isnull().sum().sum() / (len(X_train_cleaned) * len(X_train_cleaned.columns)) * 100):.2f}%")
print("=" * 90)

DETAILED MISSING VALUES ANALYSIS

FEATURES WITH MISSING VALUES:

Feature                        | Type            | Missing      | %        | Level          
-----------------------------------------------------------------------------------------------
Humidity(%)                    | Numerical       |     54,305 |   2.62% | Low (0-5%)     
Visibility(mi)                 | Numerical       |     52,679 |   2.54% | Low (0-5%)     
Weather_Condition              | Categorical     |     51,212 |   2.47% | Low (0-5%)     
Temperature(F)                 | Numerical       |     51,118 |   2.47% | Low (0-5%)     
Sunrise_Sunset                 | Categorical     |     10,216 |   0.49% | Low (0-5%)     
City                           | Categorical     |         93 |   0.00% | Low (0-5%)     

OVERALL STATISTICS:
   • Total observations: 2,072,286
   • Total features: 16
   • Features with missing: 6
   • Features without missing: 10
   • Total missing cells: 219,623
   • Overall missing rate: 0

`City` is a key location feature with very low missing rate (<1%), so we drop rows with missing City values to preserve data quality instead of imputing.

In [80]:
# Drop rows with missing City values
train_missing_city = X_train_cleaned[X_train_cleaned['City'].isnull()].index

X_train_cleaned = X_train_cleaned.drop(train_missing_city)
y_train = y_train.drop(train_missing_city)

print(f"Dropped {len(train_missing_city):,} rows with missing City from training set")
print(f"New X_train_cleaned shape: {X_train_cleaned.shape}")
print(f"New y_train shape: {y_train.shape}")

Dropped 93 rows with missing City from training set
New X_train_cleaned shape: (2072193, 16)
New y_train shape: (2492103,)


### 2.3. Create Pipeline


In [82]:
# Numerical Features
numerical_pipeline = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler())
])

In [81]:
numeric_features

['Temperature(F)', 'Humidity(%)', 'Visibility(mi)']

## 3. Outlier Processing

### 3.1. Detaild Outlier Values Analysis

## Outlier Analysis (IQR Method)
### Objective
Detect and handle abnormal values in the dataset to ensure data quality for analysis and modeling.
### Method
**IQR Calculation:**  
IQR is calculated for each numeric column:
$$
IQR = Q3 - Q1
$$
**Outlier Bounds:**  
The lower and upper bounds are:
$$
\text{Lower Bound} = Q1 - 1.5 \times IQR
$$
$$
\text{Upper Bound} = Q3 + 1.5 \times IQR
$$
Values outside this range are considered **outliers**.
### Reason for Using IQR
IQR identifies outliers based on statistical distribution, minimizing the influence of extreme values, and provides an objective method for numeric variables.
### Benefits
- Removes abnormal values, ensuring clean and reliable data.  
- Improves accuracy and stability of analysis and machine learning models.  
- Prepares data for downstream preprocessing and modeling.


In [122]:
outlier_cols = ['Vict Age']

print("\n" + "=" * 90)
print("OUTLIER ANALYSIS (IQR METHOD)")
print("=" * 90)

# Check cột tồn tại
available_cols = [c for c in outlier_cols if c in X_train_cleaned.columns]
report_data = []

for col in available_cols:
    # 1. Calculate IQR
    
    Q1 = X_train_cleaned[col].quantile(0.25)
    Q3 = X_train_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # 2. Define Limits (Tính ngưỡng thống kê ban đầu)
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    if col == 'Vict Age': 
        lower_bound = max(lower_bound, 0)    # Không âm
        upper_bound = min(upper_bound, 100)  # Không quá 100 tuổi
        
    # 3. Count outliers (Đếm dựa trên ngưỡng mới)
    outliers_mask = (X_train_cleaned[col] < lower_bound) | (X_train_cleaned[col] > upper_bound)
    num_outliers = outliers_mask.sum()
    pct_outliers = (num_outliers / len(X_train_cleaned)) * 100
    
    if num_outliers > 0:
        report_data.append({
            'Feature': col,
            'Count': num_outliers,
            'Percent': f"{pct_outliers:.2f}%",
            'Rec. Lower': f"{lower_bound:.2f}",
            'Rec. Upper': f"{upper_bound:.2f}",
            'Min': f"{X_train_cleaned[col].min():.2f}",
            'Max': f"{X_train_cleaned[col].max():.2f}"
        })

# 4. Display Report Table
if len(report_data) > 0:
    report_df = pd.DataFrame(report_data)
    print(f"Outliers detected in {len(report_data)} columns:\n")
    # Định dạng bảng đẹp
    print(report_df.to_string(index=False, justify='left'))
else:
    print("No significant outliers found using the IQR method.")

# --- OVERALL STATISTICS SECTION ---
if report_data:
    total_outlier_cells = sum(item['Count'] for item in report_data)
    total_features_analyzed = len(available_cols)
    total_cells_analyzed = len(X_train_cleaned) * total_features_analyzed

    print("\n" + "=" * 90)
    print("OVERALL OUTLIER STATISTICS:")
    print(f"   • Total observations: {len(X_train_cleaned):,}")
    print(f"   • Features analyzed: {total_features_analyzed}")
    print(f"   • Features with outliers: {len(report_data)}")
    print(f"   • Total outlier cells: {total_outlier_cells:,}")
    if total_cells_analyzed > 0:
        print(f"   • Overall outlier rate: {(total_outlier_cells / total_cells_analyzed * 100):.2f}%")
    print("=" * 90)


OUTLIER ANALYSIS (IQR METHOD)
Outliers detected in 1 columns:

Feature   Count Percent Rec. Lower Rec. Upper Min   Max   
Vict Age 100    0.01%   0.00       100.00     -4.00 120.00

OVERALL OUTLIER STATISTICS:
   • Total observations: 764,271
   • Features analyzed: 1
   • Features with outliers: 1
   • Total outlier cells: 100
   • Overall outlier rate: 0.01%


**Outlier Analysis Results**

The outlier analysis shows that only the **Vict Age** column contains outliers, and the overall rate is extremely low (0.01%).  
Some values are clearly unrealistic (e.g., -4, 120), indicating data entry errors.  
Although few in number, handling these outliers will improve data quality and ensure more reliable analysis and modeling.

### 3.2. Outlier Processing

In [123]:
class IQROutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bound_ = {}
        self.upper_bound_ = {}
        
    def fit(self, X, y=None):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
            
        for col in X.columns:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            
            lower = Q1 - self.factor * IQR
            upper = Q3 + self.factor * IQR
            
            # Giới hạn riêng cho Vict Age
            if col == 'Vict Age':
                lower = max(lower, 0)
                upper = min(upper, 100)  # Không quá 100 tuổi
                
            self.lower_bound_[col] = lower
            self.upper_bound_[col] = upper
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        if isinstance(X_copy, np.ndarray):
            X_copy = pd.DataFrame(X_copy)
            
        for col in X_copy.columns:
            if col in self.upper_bound_:
                X_copy[col] = np.clip(X_copy[col], self.lower_bound_[col], self.upper_bound_[col])
        return X_copy

## IQROutlierCapper Class

`IQROutlierCapper` is a custom scikit-learn transformer for **capping outliers** using the IQR method.  
It calculates the IQR for numeric columns, defines upper and lower bounds, and clips values outside these bounds.  
Special handling is applied for `Vict Age` to keep values within realistic limits (0–100).  

**Benefits:**  
- Reduces the impact of extreme values on analysis and models.  
- Preserves all data points while controlling outliers.  
- Easily integrates into preprocessing pipelines.

In [124]:
from sklearn import set_config
set_config(transform_output="pandas")

# Chuyển numpy sang DataFrame nếu cần
if isinstance(X_train_cleaned, np.ndarray):
    X_train_cleaned = pd.DataFrame(X_train_cleaned, columns=X_train.columns)

if isinstance(X_test_cleaned, np.ndarray):
    X_test_cleaned = pd.DataFrame(X_test_cleaned, columns=X_test.columns)

# Lọc cột tồn tại
outlier_cols = [c for c in outlier_cols if c in X_train_cleaned.columns]
print("\nCột xử lý outlier:", outlier_cols)

# Pipeline xử lý outlier
outlier_pipeline = Pipeline(steps=[
    ('capper', IQROutlierCapper(factor=1.5))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('outlier_handling', outlier_pipeline, outlier_cols)
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

# Chuyển dtype cho các cột cần
cols_to_fix = ['Premis Cd', 'Weapon Used Cd', 'Crm Cd 1', 'Crm Cd 2']
for col in cols_to_fix:
    if col in X_train_cleaned.columns:
        X_train_cleaned[col] = X_train_cleaned[col].astype('float64')
    if col in X_test_cleaned.columns:
        X_test_cleaned[col] = X_test_cleaned[col].astype('float64')

print("\n--- Đang xử lý dữ liệu (Outlier Capping)... ---")

X_train_cleaned_capped = preprocessor.fit_transform(X_train_cleaned)
X_test_cleaned_capped = preprocessor.transform(X_test_cleaned)

# Nếu output là numpy, chuyển sang DataFrame với tên cột gốc
if isinstance(X_train_cleaned_capped, np.ndarray):
    X_train_cleaned_capped = pd.DataFrame(X_train_cleaned_capped, columns=X_train_cleaned.columns)
if isinstance(X_test_cleaned_capped, np.ndarray):
    X_test_cleaned_capped = pd.DataFrame(X_test_cleaned_capped, columns=X_test_cleaned.columns)

print("Xử lý hoàn tất!")
print("-" * 90)


Cột xử lý outlier: ['Vict Age']

--- Đang xử lý dữ liệu (Outlier Capping)... ---
Xử lý hoàn tất!
------------------------------------------------------------------------------------------


## Outlier Handling Pipeline

This code implements a **pipeline to cap outliers** in the dataset using the `IQROutlierCapper` transformer.  

### Overview
- Ensures numeric columns are processed correctly by converting NumPy arrays to DataFrames.  
- Filters only the columns that exist in the dataset for outlier capping.  
- Uses a `ColumnTransformer` pipeline to apply the IQR-based outlier capping (`IQROutlierCapper`) while leaving other columns unchanged.  
- Converts specific categorical columns to `float64` to ensure compatibility with the transformer.  
- Returns cleaned datasets (`X_train_cleaned_capped`, `X_test_cleaned_capped`) as DataFrames with original column names.

**Benefits:**  
- Controls extreme values without removing any rows.  
- Maintains dataset integrity and prepares data for downstream modeling.  
- Integrates seamlessly into preprocessing workflows with scikit-learn.

In [125]:
import os

os.makedirs("dataset/processed", exist_ok=True)

# Lưu file CSV
X_train_cleaned_capped.to_csv("dataset/processed/X_train_cleaned.csv", index=False)
X_test_cleaned_capped.to_csv("dataset/processed/X_test_cleaned.csv", index=False)

## Categorical Encoder

In [ ]:
ordinal_features = ['Education Level', 'Policy Type', 'Customer Feedback', 'Exercise Frequency']
nominal_features = [col for col in categorical_features if col not in ordinal_features]

ordinal_education = ["High School", "Bachelor's", "Master's", "PhD"]
ordinal_policy = ["Basic", "Comprehensive", "Premium"]
ordinal_feedback = ["Poor", "Average", "Good"]
ordinal_exercise = ["Rarely", "Monthly", "Weekly", "Daily"]

In [ ]:
# ColumnTransformer để xử lý theo nhóm
preprocessor = ColumnTransformer(
    transformers=[
        # OneHot cho nominal categorical - ignore unknown values
        ("encode_cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_features),   
        # Ordinal Encoder - use sentinel value for unknown
        ("encode_ord", OrdinalEncoder(categories=[ordinal_education, ordinal_policy, ordinal_feedback, ordinal_exercise],
             handle_unknown='use_encoded_value',
             unknown_value=-1 
         ), 
         ordinal_features)
    ],
    remainder="passthrough"
)

X_transformed = preprocessor.fit_transform(X_train, y_train)

In [ ]:
# Lấy tên các cột sau khi transform
feature_names = preprocessor.get_feature_names_out()

print("=" * 90)
print("TRANSFORMED FEATURE NAMES")
print("=" * 90)
print(f"Total features after transformation: {len(feature_names)}\n")

# Convert transformed data to DataFrame
X_train_transformed = pd.DataFrame(X_transformed, columns=feature_names, index=X_train.index)
X_test_transformed = preprocessor.transform(X_test)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

# Add target variable back to train data
train_processed = X_train_transformed.copy()
train_processed['Premium Amount'] = y_train

print(f"✓ Train data shape: {train_processed.shape}")
print(f"✓ Test data shape: {X_test_transformed.shape}")
print(f"\nFeature names preview:")
for i, name in enumerate(feature_names[:10], 1):
    print(f"   {i}. {name}")
if len(feature_names) > 10:
    print(f"   ... and {len(feature_names) - 10} more features")
print("=" * 90)

In [ ]:
# Save processed data to CSV
import os

# Create processed directory if not exists
os.makedirs("dataset/processed", exist_ok=True)

# Save train data (with target variable)
train_processed.to_csv("dataset/processed/train_processed.csv", index=False)
print(f"✓ Saved: dataset/processed/train_processed.csv ({train_processed.shape})")

# Save test data (without target variable)
X_test_transformed.to_csv("dataset/processed/test_processed.csv", index=False)
print(f"✓ Saved: dataset/processed/test_processed.csv ({X_test_transformed.shape})")

print("\n" + "=" * 90)
print("ALL FILES SAVED SUCCESSFULLY!")
print(f"   • Train: {train_processed.shape[0]:,} rows × {train_processed.shape[1]} columns")
print(f"   • Test:  {X_test_transformed.shape[0]:,} rows × {X_test_transformed.shape[1]} columns")
print("=" * 90)